In [32]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict

In [33]:
load_dotenv()
model =  ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=1.0)

In [34]:
class BLOGstate(TypedDict):
    topic: str
    outline: str
    content: str
    rating: float

In [35]:
def outline(state: BLOGstate) -> BLOGstate:
    topic = state['topic']
    prompt = f'Generate an outline for a blog based on this topic - {topic}'
    state['outline'] = model.invoke(prompt).content
    return state

In [36]:
def blog(state: BLOGstate) -> BLOGstate:
    topic = state['topic']
    outline = state['outline']
    prompt = f'Generate a detailed blog for this topic - {topic}, based on this outline: \n {outline}'
    state['content'] = model.invoke(prompt).content
    return state

In [37]:
def evaluator(state: BLOGstate) -> BLOGstate:
    outline = state['outline']
    content = state['content']
    prompt = f'Evaluate and ONLY generate a rating between 1 to 10 for this blog \n - {content}, based on this outline: \n {outline}, THE FINAL RESULT SHOULD JUST BE THE RATING NUMBER'
    state['rating'] = model.invoke(prompt).content
    return state

In [38]:
graph = StateGraph(BLOGstate)
graph.add_node('outline', outline)
graph.add_node('blog', blog)
graph.add_node('evaluator', evaluator)

graph.add_edge(START,'outline')
graph.add_edge('outline','blog')
graph.add_edge('blog', 'evaluator')
graph.add_edge('evaluator', END)

workflow = graph.compile()


In [39]:
initial_state = {'topic': 'Messi vs Ronaldo'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'topic': 'Messi vs Ronaldo', 'outline': '## Blog Post Outline: Messi vs. Ronaldo - The Greatest Debate in Football History\n\n**Target Audience:** Football fans, sports enthusiasts, casual readers interested in sports rivalries.\n\n**Goal:** To explore the Messi vs. Ronaldo debate, analyze their careers, highlight their unique strengths, and offer a balanced perspective on who could be considered "the greatest."\n\n---\n\n### **Blog Post Title Options:**\n\n*   Messi vs. Ronaldo: The Ultimate Showdown for Football Supremacy\n*   Beyond the Goals: Unpacking the Messi vs. Ronaldo Rivalry\n*   GOAT or GOATs? A Deep Dive into Messi and Ronaldo\'s Legacy\n*   The Enduring Debate: Why Messi vs. Ronaldo Still Captivates the World\n\n---\n\n### **I. Introduction: The Rivalry That Defined an Era**\n\n*   **A. Hook:** Start with a captivating statement about the immense impact and longevity of their rivalry. Mention the decades of dominance.\n*   **B. Introduce the Players:** Briefly introduce 

In [40]:
print(final_state['outline'])

## Blog Post Outline: Messi vs. Ronaldo - The Greatest Debate in Football History

**Target Audience:** Football fans, sports enthusiasts, casual readers interested in sports rivalries.

**Goal:** To explore the Messi vs. Ronaldo debate, analyze their careers, highlight their unique strengths, and offer a balanced perspective on who could be considered "the greatest."

---

### **Blog Post Title Options:**

*   Messi vs. Ronaldo: The Ultimate Showdown for Football Supremacy
*   Beyond the Goals: Unpacking the Messi vs. Ronaldo Rivalry
*   GOAT or GOATs? A Deep Dive into Messi and Ronaldo's Legacy
*   The Enduring Debate: Why Messi vs. Ronaldo Still Captivates the World

---

### **I. Introduction: The Rivalry That Defined an Era**

*   **A. Hook:** Start with a captivating statement about the immense impact and longevity of their rivalry. Mention the decades of dominance.
*   **B. Introduce the Players:** Briefly introduce Lionel Messi and Cristiano Ronaldo as the two titans of modern 

In [41]:
print(final_state['content'])

## Messi vs. Ronaldo: The Ultimate Showdown for Football Supremacy

For over a decade, the world of football has been captivated by a rivalry unlike any other. Two titans, Lionel Messi and Cristiano Ronaldo, have not only dominated the sport but have also etched their names into the annals of sporting history. Their battles on the pitch, their individual brilliance, and their relentless pursuit of perfection have ignited a debate that continues to rage: **Who is the greatest footballer of all time?**

This isn't just a debate for die-hard fans; it's a conversation that transcends the sport, touching upon themes of natural talent versus sheer dedication, artistry versus athleticism, and the very definition of greatness. For years, we've watched them redefine what's possible, shattering records and pushing each other to unimaginable heights. In this deep dive, we'll unpack their incredible journeys, dissect their unique styles, compare their monumental achievements, and explore the argum

In [42]:
print(final_state['rating'])

9
